[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tsilva/aiml-notebooks/blob/main/notebooks/image-generation.ipynb)

# Image Generation: VAEs and GANs

**Learning Objectives:**
- Understand the fundamentals of generative modeling for images
- Learn Variational Autoencoder (VAE) architecture and theory
- Learn Generative Adversarial Network (GAN) architecture and training
- Implement sampling and generation strategies
- Explore latent space manipulation and interpolation
- Compare different generative modeling approaches

**What We'll Build:**
1. A VAE that learns to generate MNIST digits
2. A DCGAN (Deep Convolutional GAN) that generates MNIST digits
3. Tools for exploring and manipulating the learned latent spaces
4. Comparative analysis of both approaches

## Part 1: Introduction to Image Generation

### What is Image Generation?

Image generation is the task of creating new, realistic images from scratch. Unlike discriminative models (which classify or detect), **generative models** learn the underlying distribution of training data and can sample new instances.

### Why Image Generation?

**Applications:**
- **Data Augmentation**: Generate synthetic training data
- **Creative Tools**: Art generation, design assistance
- **Content Creation**: Video games, movies, advertising
- **Anomaly Detection**: Model normal data, detect outliers
- **Compression**: Learn compact representations
- **Super-Resolution**: Enhance low-resolution images

### Key Challenges:

1. **High Dimensionality**: Images have thousands/millions of pixels
2. **Complex Dependencies**: Pixels are highly correlated
3. **Mode Collapse**: Model generates limited variety
4. **Evaluation**: Hard to measure generation quality objectively
5. **Training Stability**: Some approaches are difficult to train

## Part 2: Setup and Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
from tqdm.auto import tqdm
import time

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Configure device (MPS for Mac, CUDA for GPU, CPU otherwise)
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using MPS (Metal Performance Shaders) device")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using CUDA device: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("Using CPU device")

print(f"PyTorch version: {torch.__version__}")

## Part 3: Load and Explore MNIST Dataset

We'll use MNIST (handwritten digits) as our training dataset. It's a good starting point because:
- Relatively simple (28x28 grayscale images)
- Fast to train
- Easy to evaluate generation quality visually
- Well-understood benchmark

In [ ]:
# Data transforms: convert to tensor and normalize to [-1, 1] range
# Note: We use [-1, 1] instead of [0, 1] as it often works better with neural networks
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))  # (mean, std) for normalization
])

# Download and load training data
train_dataset = datasets.MNIST(
    root='./tmp/data',
    train=True,
    download=True,
    transform=transform
)

# Download and load test data
test_dataset = datasets.MNIST(
    root='./tmp/data',
    train=False,
    download=True,
    transform=transform
)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Image shape: {train_dataset[0][0].shape}")
print(f"Number of classes: {len(train_dataset.classes)}")

In [ ]:
# Visualize sample images
def show_images(images, title="Images", nrow=8, figsize=(12, 6)):
    """Display a grid of images."""
    # Denormalize from [-1, 1] to [0, 1] for display
    images = images * 0.5 + 0.5
    grid = make_grid(images, nrow=nrow, padding=2)
    
    plt.figure(figsize=figsize)
    plt.imshow(grid.permute(1, 2, 0).cpu().numpy(), cmap='gray')
    plt.title(title)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

# Show random training samples
sample_indices = np.random.choice(len(train_dataset), 64, replace=False)
sample_images = torch.stack([train_dataset[i][0] for i in sample_indices])
sample_labels = [train_dataset[i][1] for i in sample_indices]

show_images(sample_images, "Random MNIST Training Samples")

## Part 4: Generative Modeling Overview

### What is a Generative Model?

A generative model learns the probability distribution $P(X)$ of the training data $X$. Once learned, we can:
- **Sample**: Generate new data points $x \sim P(X)$
- **Evaluate**: Compute probability $P(x)$ for a given $x$
- **Manipulate**: Explore the learned distribution

### Two Main Approaches:

#### 1. Explicit Density Models (VAE)
- Explicitly define and optimize $P(X)$
- **Pros**: Principled probabilistic framework, stable training
- **Cons**: May produce blurry images, limited flexibility
- **Examples**: VAE, PixelCNN, Normalizing Flows

#### 2. Implicit Density Models (GAN)
- Learn to generate samples without explicitly modeling $P(X)$
- **Pros**: Sharp, realistic images
- **Cons**: Difficult to train, mode collapse issues
- **Examples**: GAN, DCGAN, StyleGAN

### Latent Variable Models

Both VAE and GAN use **latent variables** $z$:
- $z$ is a low-dimensional representation (e.g., 100D)
- $x$ is high-dimensional data (e.g., 784D for MNIST)
- Goal: Learn mapping $z \rightarrow x$ where similar $z$ produce similar $x$

**Why latent variables?**
- Dimensionality reduction (100D vs 784D)
- Structured representation (smooth interpolation)
- Computational efficiency

### Reflection Question:

Before we dive into implementations, consider:
- What makes a "good" generated image?
- How would you measure whether a generative model has learned the data distribution?
- What challenges might arise when trying to generate images?

## Part 5: Variational Autoencoders (VAE) - Theory

### Autoencoder Basics

A standard **autoencoder** has two parts:
1. **Encoder**: $E(x) = z$ (compress to latent code)
2. **Decoder**: $D(z) = \hat{x}$ (reconstruct from latent code)

**Training objective**: Minimize reconstruction loss $\|x - \hat{x}\|^2$

**Problem**: Standard autoencoders can't generate new images!
- Latent space is irregular and discontinuous
- Random $z$ values produce garbage
- Only works for encoding/decoding seen data

### Variational Autoencoder (VAE)

**Key Innovation**: Force latent space to follow a **prior distribution** (typically $\mathcal{N}(0, I)$)

#### Encoder
Instead of encoding to a single point $z$, encode to a **distribution**:
$$q_\phi(z|x) = \mathcal{N}(\mu(x), \sigma^2(x))$$

The encoder outputs:
- $\mu(x)$: Mean vector
- $\log \sigma^2(x)$: Log-variance vector (for numerical stability)

#### Reparameterization Trick

**Problem**: Can't backpropagate through random sampling!

**Solution**: Reparameterize sampling as:
$$z = \mu + \sigma \odot \epsilon, \quad \epsilon \sim \mathcal{N}(0, I)$$

Now randomness is in $\epsilon$ (independent of parameters), so we can backprop through $\mu$ and $\sigma$.

#### Decoder
Generates reconstruction from sampled $z$:
$$p_\theta(x|z)$$

### VAE Loss Function: ELBO

VAE optimizes the **Evidence Lower Bound (ELBO)**:

$$\mathcal{L} = \underbrace{\mathbb{E}_{q_\phi(z|x)}[\log p_\theta(x|z)]}_\text{Reconstruction Loss} - \underbrace{D_{KL}(q_\phi(z|x) \| p(z))}_\text{KL Divergence}$$

#### Component 1: Reconstruction Loss
- Measures how well decoder reconstructs input
- Typically: MSE loss or Binary Cross-Entropy
- **Goal**: Make $\hat{x}$ similar to $x$

#### Component 2: KL Divergence
- Measures how much $q_\phi(z|x)$ differs from prior $p(z) = \mathcal{N}(0, I)$
- Regularizes latent space to be continuous and structured
- **Closed form** for diagonal Gaussians:

$$D_{KL} = -\frac{1}{2} \sum_{j=1}^J (1 + \log \sigma_j^2 - \mu_j^2 - \sigma_j^2)$$

where $J$ is the latent dimension.

### Why KL Divergence?

The KL term forces:
1. **Continuity**: Similar points in latent space decode to similar images
2. **Completeness**: All points in latent space are valid (no "holes")
3. **Generalization**: Can sample $z \sim \mathcal{N}(0, I)$ to generate new images

### Training Trade-off

VAE balances two competing objectives:
- **Reconstruction**: Wants to encode all information in $z$
- **Regularization**: Wants $z$ to be simple (close to $\mathcal{N}(0, I)$)

This creates a **compression** effect: encoder must learn efficient representations.

### Reflection Question:

- Why do we need the reparameterization trick? What would happen without it?
- What would happen if we removed the KL divergence term?
- Why use $\log \sigma^2$ instead of $\sigma^2$ directly?

## Part 6: VAE Architecture Implementation

Now let's implement a VAE for MNIST. We'll use a convolutional architecture for better image modeling.

In [ ]:
class VAEEncoder(nn.Module):
    """VAE Encoder: x -> (mu, log_var)"""
    
    def __init__(self, latent_dim=20):
        super().__init__()
        self.latent_dim = latent_dim
        
        # Convolutional layers for feature extraction
        # Input: (1, 28, 28)
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1)  # -> (32, 14, 14)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1) # -> (64, 7, 7)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1) # -> (128, 4, 4)
        
        # Fully connected layers to latent space
        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.fc_mu = nn.Linear(256, latent_dim)
        self.fc_log_var = nn.Linear(256, latent_dim)
        
    def forward(self, x):
        # Convolutional layers with ReLU activation
        h = F.relu(self.conv1(x))
        h = F.relu(self.conv2(h))
        h = F.relu(self.conv3(h))
        
        # Flatten
        h = h.view(h.size(0), -1)
        
        # Fully connected layers
        h = F.relu(self.fc1(h))
        
        # Output: mean and log-variance vectors
        mu = self.fc_mu(h)
        log_var = self.fc_log_var(h)
        
        return mu, log_var


class VAEDecoder(nn.Module):
    """VAE Decoder: z -> x_hat"""
    
    def __init__(self, latent_dim=20):
        super().__init__()
        self.latent_dim = latent_dim
        
        # Fully connected layers from latent space
        self.fc1 = nn.Linear(latent_dim, 256)
        self.fc2 = nn.Linear(256, 128 * 4 * 4)
        
        # Transposed convolutional layers (upsampling)
        # Input: (128, 4, 4)
        self.deconv1 = nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1) # -> (64, 8, 8)
        self.deconv2 = nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1)  # -> (32, 16, 16)
        self.deconv3 = nn.ConvTranspose2d(32, 1, kernel_size=4, stride=2, padding=3)   # -> (1, 28, 28)
        
    def forward(self, z):
        # Fully connected layers
        h = F.relu(self.fc1(z))
        h = F.relu(self.fc2(h))
        
        # Reshape to feature maps
        h = h.view(h.size(0), 128, 4, 4)
        
        # Transposed convolutions with ReLU
        h = F.relu(self.deconv1(h))
        h = F.relu(self.deconv2(h))
        
        # Final layer with tanh to match [-1, 1] range
        x_hat = torch.tanh(self.deconv3(h))
        
        return x_hat


class VAE(nn.Module):
    """Complete VAE model combining encoder and decoder."""
    
    def __init__(self, latent_dim=20):
        super().__init__()
        self.latent_dim = latent_dim
        
        self.encoder = VAEEncoder(latent_dim)
        self.decoder = VAEDecoder(latent_dim)
        
    def reparameterize(self, mu, log_var):
        """Reparameterization trick: z = mu + sigma * epsilon"""
        std = torch.exp(0.5 * log_var)  # Standard deviation
        eps = torch.randn_like(std)      # Sample epsilon ~ N(0, 1)
        z = mu + std * eps               # Reparameterize
        return z
    
    def forward(self, x):
        # Encode to latent distribution
        mu, log_var = self.encoder(x)
        
        # Sample latent code using reparameterization trick
        z = self.reparameterize(mu, log_var)
        
        # Decode to reconstruction
        x_hat = self.decoder(z)
        
        return x_hat, mu, log_var
    
    def generate(self, num_samples=64):
        """Generate new images by sampling from prior."""
        with torch.no_grad():
            # Sample from standard normal
            z = torch.randn(num_samples, self.latent_dim).to(next(self.parameters()).device)
            # Decode
            images = self.decoder(z)
        return images


# Create model
vae = VAE(latent_dim=20).to(device)

# Count parameters
total_params = sum(p.numel() for p in vae.parameters())
trainable_params = sum(p.numel() for p in vae.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"\nModel architecture:")
print(vae)

### Architecture Notes:

**Encoder:**
- Three conv layers progressively downsample (28→14→7→4)
- Channels increase with depth (1→32→64→128)
- Final FC layers produce $\mu$ and $\log \sigma^2$

**Decoder:**
- Mirror architecture of encoder
- Transposed convolutions upsample (4→8→16→28)
- `tanh` activation matches data range [-1, 1]

**Latent Space:**
- 20 dimensions (highly compressed from 784)
- Can adjust based on complexity/quality trade-off

## Part 7: VAE Loss Function

Let's implement the ELBO loss with both reconstruction and KL divergence terms.

In [ ]:
def vae_loss(x, x_hat, mu, log_var, beta=1.0):
    """
    VAE loss = Reconstruction Loss + beta * KL Divergence
    
    Args:
        x: Original images
        x_hat: Reconstructed images
        mu: Latent mean
        log_var: Latent log-variance
        beta: Weight for KL term (beta-VAE)
    
    Returns:
        total_loss, recon_loss, kl_loss
    """
    # Reconstruction loss (MSE)
    # Sum over all dimensions, average over batch
    recon_loss = F.mse_loss(x_hat, x, reduction='sum') / x.size(0)
    
    # KL divergence loss
    # Closed form: -0.5 * sum(1 + log(sigma^2) - mu^2 - sigma^2)
    # Sum over latent dimensions, average over batch
    kl_loss = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp()) / x.size(0)
    
    # Total loss (ELBO)
    total_loss = recon_loss + beta * kl_loss
    
    return total_loss, recon_loss, kl_loss


# Test the loss function
test_batch = next(iter(DataLoader(train_dataset, batch_size=32)))[0].to(device)
test_recon, test_mu, test_log_var = vae(test_batch)
test_loss, test_recon_loss, test_kl_loss = vae_loss(test_batch, test_recon, test_mu, test_log_var)

print(f"Test loss components:")
print(f"  Total loss: {test_loss.item():.4f}")
print(f"  Reconstruction loss: {test_recon_loss.item():.4f}")
print(f"  KL divergence: {test_kl_loss.item():.4f}")

### Beta-VAE

The parameter $\beta$ controls the trade-off:
- **$\beta = 1$**: Standard VAE (balance reconstruction and regularization)
- **$\beta > 1$**: Emphasize disentanglement (more structured latent space)
- **$\beta < 1$**: Emphasize reconstruction quality (sharper images)

We'll start with $\beta = 1$ (standard VAE).

## Part 8: Training the VAE

Let's train the VAE on MNIST and monitor the learning progress.

In [ ]:
# Training configuration
BATCH_SIZE = 128
LEARNING_RATE = 1e-3
NUM_EPOCHS = 20
BETA = 1.0

# Create data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,  # Set to 0 for MPS compatibility
    pin_memory=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

# Initialize model and optimizer
vae = VAE(latent_dim=20).to(device)
optimizer = optim.Adam(vae.parameters(), lr=LEARNING_RATE)

print(f"Configuration:")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Beta: {BETA}")
print(f"  Training batches: {len(train_loader)}")
print(f"  Test batches: {len(test_loader)}")

In [ ]:
def train_vae_epoch(model, loader, optimizer, beta=1.0):
    """Train VAE for one epoch."""
    model.train()
    
    total_loss = 0
    total_recon = 0
    total_kl = 0
    
    for batch_idx, (images, _) in enumerate(tqdm(loader, desc="Training", leave=False)):
        images = images.to(device)
        
        # Forward pass
        x_hat, mu, log_var = model(images)
        
        # Compute loss
        loss, recon_loss, kl_loss = vae_loss(images, x_hat, mu, log_var, beta=beta)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Accumulate losses
        total_loss += loss.item()
        total_recon += recon_loss.item()
        total_kl += kl_loss.item()
    
    # Average losses
    avg_loss = total_loss / len(loader)
    avg_recon = total_recon / len(loader)
    avg_kl = total_kl / len(loader)
    
    return avg_loss, avg_recon, avg_kl


def evaluate_vae(model, loader, beta=1.0):
    """Evaluate VAE on test set."""
    model.eval()
    
    total_loss = 0
    total_recon = 0
    total_kl = 0
    
    with torch.no_grad():
        for images, _ in loader:
            images = images.to(device)
            
            # Forward pass
            x_hat, mu, log_var = model(images)
            
            # Compute loss
            loss, recon_loss, kl_loss = vae_loss(images, x_hat, mu, log_var, beta=beta)
            
            # Accumulate losses
            total_loss += loss.item()
            total_recon += recon_loss.item()
            total_kl += kl_loss.item()
    
    # Average losses
    avg_loss = total_loss / len(loader)
    avg_recon = total_recon / len(loader)
    avg_kl = total_kl / len(loader)
    
    return avg_loss, avg_recon, avg_kl

In [ ]:
# Training loop with visualization
history = {
    'train_loss': [],
    'train_recon': [],
    'train_kl': [],
    'test_loss': [],
    'test_recon': [],
    'test_kl': []
}

# Fixed test batch for visualization
fixed_test_images = next(iter(test_loader))[0][:64].to(device)

print("Training VAE...\n")
start_time = time.time()

for epoch in range(NUM_EPOCHS):
    # Train
    train_loss, train_recon, train_kl = train_vae_epoch(vae, train_loader, optimizer, beta=BETA)
    
    # Evaluate
    test_loss, test_recon, test_kl = evaluate_vae(vae, test_loader, beta=BETA)
    
    # Store history
    history['train_loss'].append(train_loss)
    history['train_recon'].append(train_recon)
    history['train_kl'].append(train_kl)
    history['test_loss'].append(test_loss)
    history['test_recon'].append(test_recon)
    history['test_kl'].append(test_kl)
    
    # Print progress
    elapsed = time.time() - start_time
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} ({elapsed:.1f}s)")
    print(f"  Train - Loss: {train_loss:.4f}, Recon: {train_recon:.4f}, KL: {train_kl:.4f}")
    print(f"  Test  - Loss: {test_loss:.4f}, Recon: {test_recon:.4f}, KL: {test_kl:.4f}")
    
    # Visualize progress every 5 epochs
    if (epoch + 1) % 5 == 0 or epoch == 0:
        vae.eval()
        with torch.no_grad():
            # Reconstruct test images
            recon_images, _, _ = vae(fixed_test_images)
            
            # Generate new images
            generated_images = vae.generate(64)
        
        # Show reconstructions and generations
        fig, axes = plt.subplots(2, 2, figsize=(12, 12))
        
        # Original images
        orig_grid = make_grid(fixed_test_images[:64] * 0.5 + 0.5, nrow=8, padding=2)
        axes[0, 0].imshow(orig_grid.permute(1, 2, 0).cpu().numpy(), cmap='gray')
        axes[0, 0].set_title("Original Test Images")
        axes[0, 0].axis('off')
        
        # Reconstructed images
        recon_grid = make_grid(recon_images[:64] * 0.5 + 0.5, nrow=8, padding=2)
        axes[0, 1].imshow(recon_grid.permute(1, 2, 0).cpu().numpy(), cmap='gray')
        axes[0, 1].set_title("Reconstructed Images")
        axes[0, 1].axis('off')
        
        # Generated images
        gen_grid = make_grid(generated_images * 0.5 + 0.5, nrow=8, padding=2)
        axes[1, 0].imshow(gen_grid.permute(1, 2, 0).cpu().numpy(), cmap='gray')
        axes[1, 0].set_title("Generated Images (from prior)")
        axes[1, 0].axis('off')
        
        # Training curves
        axes[1, 1].plot(history['train_loss'], label='Train Loss', alpha=0.7)
        axes[1, 1].plot(history['test_loss'], label='Test Loss', alpha=0.7)
        axes[1, 1].plot(history['train_recon'], label='Train Recon', alpha=0.7, linestyle='--')
        axes[1, 1].plot(history['train_kl'], label='Train KL', alpha=0.7, linestyle='--')
        axes[1, 1].set_xlabel('Epoch')
        axes[1, 1].set_ylabel('Loss')
        axes[1, 1].set_title('Training Progress')
        axes[1, 1].legend()
        axes[1, 1].grid(True, alpha=0.3)
        
        plt.suptitle(f"VAE Training - Epoch {epoch+1}/{NUM_EPOCHS}", fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()

total_time = time.time() - start_time
print(f"\nTraining complete in {total_time:.1f}s ({total_time/60:.1f} minutes)")
print(f"Final test loss: {history['test_loss'][-1]:.4f}")

### Training Observations:

**What to look for:**
1. **Reconstruction quality**: Should improve steadily
2. **KL divergence**: Should stabilize (not collapse to 0)
3. **Generated samples**: Should become recognizable digits
4. **Overfitting**: Train/test gap (should be small)

**Common issues:**
- **KL vanishing**: If KL → 0, model ignores latent code (posterior collapse)
- **Blurry reconstructions**: VAEs tend to produce blurry images (averaging effect)
- **Poor generations**: May need more epochs or larger latent dim

## Part 9: Exploring the Latent Space

One of the most powerful features of VAEs is the structured latent space. Let's explore it!

In [ ]:
# Sample and visualize generated images
vae.eval()
with torch.no_grad():
    generated = vae.generate(64)

show_images(generated, "Random Samples from VAE")

### Latent Space Interpolation

Since the latent space is continuous, we can **interpolate** between two images:
1. Encode two images to get $z_1$ and $z_2$
2. Create intermediate points: $z_t = (1-t) z_1 + t z_2$ for $t \in [0, 1]$
3. Decode each $z_t$ to see smooth transitions

In [ ]:
def interpolate_latent(model, img1, img2, num_steps=10):
    """Interpolate between two images in latent space."""
    model.eval()
    
    with torch.no_grad():
        # Encode images to latent space
        mu1, log_var1 = model.encoder(img1.unsqueeze(0))
        mu2, log_var2 = model.encoder(img2.unsqueeze(0))
        
        # Use means (no sampling) for stable interpolation
        z1 = mu1
        z2 = mu2
        
        # Create interpolation steps
        alphas = torch.linspace(0, 1, num_steps).to(device)
        interpolations = []
        
        for alpha in alphas:
            # Linear interpolation
            z_interp = (1 - alpha) * z1 + alpha * z2
            
            # Decode
            img_interp = model.decoder(z_interp)
            interpolations.append(img_interp)
        
        return torch.cat(interpolations, dim=0)


# Select pairs of images to interpolate
test_images = [test_dataset[i][0] for i in [0, 15, 27, 35]]  # Select different digits

# Perform interpolations
fig, axes = plt.subplots(2, 1, figsize=(15, 6))

for idx in range(2):
    img1 = test_images[idx*2].to(device)
    img2 = test_images[idx*2 + 1].to(device)
    
    interpolated = interpolate_latent(vae, img1, img2, num_steps=12)
    grid = make_grid(interpolated * 0.5 + 0.5, nrow=12, padding=2)
    
    axes[idx].imshow(grid.permute(1, 2, 0).cpu().numpy(), cmap='gray')
    axes[idx].set_title(f"Interpolation {idx+1}: Start → End")
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

### Latent Space Arithmetic

We can perform arithmetic in latent space:
- Find "directions" that correspond to features
- Move along these directions to modify attributes
- Example: find "angle" direction, adjust rotation

In [ ]:
# Explore random walks in latent space
vae.eval()

with torch.no_grad():
    # Start from a random point
    z_start = torch.randn(1, 20).to(device)
    
    # Random direction
    direction = torch.randn(1, 20).to(device)
    direction = direction / direction.norm()  # Normalize
    
    # Walk along direction
    steps = torch.linspace(-3, 3, 12).to(device)
    walk = []
    
    for step in steps:
        z = z_start + step * direction
        img = vae.decoder(z)
        walk.append(img)
    
    walk_images = torch.cat(walk, dim=0)

show_images(walk_images, "Random Walk in Latent Space (-3 to +3)", nrow=12, figsize=(15, 3))

### Reflection Question:

- What do you observe in the interpolations? Are transitions smooth?
- Does the latent space seem well-structured?
- What happens in the random walk? Do you see gradual changes?

## Part 10: Generative Adversarial Networks (GAN) - Theory

### The GAN Framework

GANs take a completely different approach from VAEs. Instead of explicitly modeling $P(X)$, GANs use a **game-theoretic** framework.

### Two Players:

#### 1. Generator (G)
- Input: Random noise $z \sim P(z)$ (e.g., $\mathcal{N}(0, I)$)
- Output: Fake image $\hat{x} = G(z)$
- Goal: **Fool** the discriminator (generate realistic images)

#### 2. Discriminator (D)
- Input: Image $x$ (real or fake)
- Output: Probability $D(x) \in [0, 1]$ that $x$ is real
- Goal: **Distinguish** real from fake images

### Training Dynamics

GAN training is a **minimax game**:

$$\min_G \max_D \mathbb{E}_{x \sim p_{data}}[\log D(x)] + \mathbb{E}_{z \sim p(z)}[\log(1 - D(G(z)))]$$

**In plain English:**
- **Discriminator** maximizes: correctly classifying real as real (maximize $\log D(x)$) and fake as fake (maximize $\log(1-D(G(z)))$)
- **Generator** minimizes: fooling discriminator (minimize $\log(1-D(G(z)))$, i.e., make $D(G(z)) \rightarrow 1$)

### Training Algorithm

Alternate between:
1. **Update D**: Train discriminator to distinguish real from fake
   - Sample real batch $x \sim p_{data}$
   - Sample noise $z \sim p(z)$, generate fake batch $\hat{x} = G(z)$
   - Update D to maximize $\log D(x) + \log(1 - D(\hat{x}))$

2. **Update G**: Train generator to fool discriminator
   - Sample noise $z \sim p(z)$
   - Update G to maximize $\log D(G(z))$ (minimize $\log(1 - D(G(z)))$)

### Why This Works

**Equilibrium theory**: At convergence (Nash equilibrium):
- $D(x) = 0.5$ for all $x$ (can't tell real from fake)
- $P_G = P_{data}$ (generator distribution matches data)

**Intuition**: 
- D provides learning signal (gradient) to G
- As G improves, D must work harder
- Competition drives both to improve

### Advantages of GANs

1. **Sharp images**: No averaging, can generate fine details
2. **Implicit modeling**: Don't need to model $P(X)$ explicitly
3. **Flexible architectures**: Can use any differentiable G and D

### Challenges of GANs

1. **Training instability**: Difficult to balance G and D
2. **Mode collapse**: G generates limited variety
3. **Evaluation**: Hard to measure quality objectively
4. **Hyperparameter sensitivity**: Requires careful tuning

## Part 11: DCGAN Architecture - Theory

**DCGAN (Deep Convolutional GAN)** introduced architectural guidelines that stabilized GAN training.

### Key Innovations:

1. **Replace pooling with strided convolutions**
   - Discriminator: strided convolutions (downsampling)
   - Generator: transposed convolutions (upsampling)
   - Lets network learn its own downsampling/upsampling

2. **Use batch normalization**
   - Stabilizes training
   - Helps gradients flow
   - Exception: Don't use in generator output and discriminator input

3. **Remove fully connected layers**
   - Use global pooling instead
   - Deeper architectures

4. **Activation functions**
   - Generator: ReLU (hidden), Tanh (output)
   - Discriminator: LeakyReLU (all layers)

### Architecture Pattern:

**Generator**: $z \rightarrow x$
```
Input: z (e.g., 100D noise)
↓ Project & reshape (e.g., to 7×7×256)
↓ TransposeConv + BatchNorm + ReLU (7×7 → 14×14)
↓ TransposeConv + BatchNorm + ReLU (14×14 → 28×28)
↓ TransposeConv + Tanh (28×28 → 28×28)
Output: Image (1×28×28)
```

**Discriminator**: $x \rightarrow$ real/fake
```
Input: Image (1×28×28)
↓ Conv + LeakyReLU (28×28 → 14×14)
↓ Conv + BatchNorm + LeakyReLU (14×14 → 7×7)
↓ Conv + BatchNorm + LeakyReLU (7×7 → 4×4)
↓ Conv + Sigmoid (4×4 → 1)
Output: Probability (real/fake)
```

### Training Tips:

1. **Learning rate**: Usually 0.0002
2. **Optimizer**: Adam with $\beta_1 = 0.5$, $\beta_2 = 0.999$
3. **Label smoothing**: Use 0.9 instead of 1.0 for real labels
4. **Update ratio**: Often update D more than G (e.g., 2:1 or 5:1)
5. **Latent dimension**: Typically 100-512

### Reflection Question:

- Why does GAN produce sharper images than VAE?
- What could go wrong if D becomes too strong compared to G?
- What could go wrong if G becomes too strong compared to D?

## Part 12: DCGAN Implementation

Let's implement DCGAN following the architecture guidelines.

In [ ]:
class DCGANGenerator(nn.Module):
    """DCGAN Generator: z -> image"""
    
    def __init__(self, latent_dim=100, output_channels=1):
        super().__init__()
        self.latent_dim = latent_dim
        
        # Initial projection and reshape
        self.fc = nn.Linear(latent_dim, 256 * 7 * 7)
        
        # Transposed convolutions (upsampling)
        self.deconv_layers = nn.Sequential(
            # Input: (256, 7, 7)
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            
            # (256, 7, 7) -> (128, 14, 14)
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            
            # (128, 14, 14) -> (64, 28, 28)
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(True),
            
            # (64, 28, 28) -> (1, 28, 28)
            nn.Conv2d(64, output_channels, kernel_size=3, stride=1, padding=1),
            nn.Tanh()  # Output in [-1, 1] to match data normalization
        )
    
    def forward(self, z):
        # Project and reshape
        h = self.fc(z)
        h = h.view(h.size(0), 256, 7, 7)
        
        # Apply transposed convolutions
        img = self.deconv_layers(h)
        
        return img


class DCGANDiscriminator(nn.Module):
    """DCGAN Discriminator: image -> real/fake"""
    
    def __init__(self, input_channels=1):
        super().__init__()
        
        self.conv_layers = nn.Sequential(
            # Input: (1, 28, 28)
            
            # (1, 28, 28) -> (64, 14, 14)
            nn.Conv2d(input_channels, 64, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            
            # (64, 14, 14) -> (128, 7, 7)
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            
            # (128, 7, 7) -> (256, 3, 3)
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            
            # (256, 3, 3) -> (1, 1, 1)
            nn.Conv2d(256, 1, kernel_size=3, stride=1, padding=0),
            nn.Sigmoid()  # Output probability
        )
    
    def forward(self, img):
        # Apply convolutions
        validity = self.conv_layers(img)
        
        # Flatten to scalar
        validity = validity.view(validity.size(0), -1)
        
        return validity


# Initialize models
latent_dim = 100
generator = DCGANGenerator(latent_dim=latent_dim).to(device)
discriminator = DCGANDiscriminator().to(device)

# Count parameters
g_params = sum(p.numel() for p in generator.parameters())
d_params = sum(p.numel() for p in discriminator.parameters())

print(f"Generator parameters: {g_params:,}")
print(f"Discriminator parameters: {d_params:,}")
print(f"\nGenerator architecture:")
print(generator)
print(f"\nDiscriminator architecture:")
print(discriminator)

### Weight Initialization

DCGAN paper recommends initializing weights from $\mathcal{N}(0, 0.02)$ for better training stability.

In [ ]:
def weights_init(m):
    """Initialize network weights (DCGAN paper recommendation)."""
    classname = m.__class__.__name__
    
    if classname.find('Conv') != -1:
        # Initialize conv layers
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        # Initialize batch norm layers
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

# Apply weight initialization
generator.apply(weights_init)
discriminator.apply(weights_init)

print("Weights initialized")

## Part 13: GAN Loss Functions

Let's implement the binary cross-entropy loss for both generator and discriminator.

In [ ]:
# Loss function
adversarial_loss = nn.BCELoss()

# Optimizers (DCGAN paper: lr=0.0002, beta1=0.5)
lr = 0.0002
beta1 = 0.5
beta2 = 0.999

optimizer_G = optim.Adam(generator.parameters(), lr=lr, betas=(beta1, beta2))
optimizer_D = optim.Adam(discriminator.parameters(), lr=lr, betas=(beta1, beta2))

print(f"Optimizers configured:")
print(f"  Learning rate: {lr}")
print(f"  Adam betas: ({beta1}, {beta2})")

In [ ]:
def train_discriminator(real_images, generator, discriminator, optimizer, loss_fn, latent_dim, device, label_smoothing=True):
    """Train discriminator for one step."""
    batch_size = real_images.size(0)
    
    # Create labels (with optional label smoothing)
    if label_smoothing:
        real_labels = torch.ones(batch_size, 1).to(device) * 0.9  # Smooth to 0.9 instead of 1.0
    else:
        real_labels = torch.ones(batch_size, 1).to(device)
    fake_labels = torch.zeros(batch_size, 1).to(device)
    
    # ---------------------
    # Train on real images
    # ---------------------
    optimizer.zero_grad()
    
    real_pred = discriminator(real_images)
    real_loss = loss_fn(real_pred, real_labels)
    
    # ---------------------
    # Train on fake images
    # ---------------------
    z = torch.randn(batch_size, latent_dim).to(device)
    fake_images = generator(z).detach()  # Detach to not train generator
    fake_pred = discriminator(fake_images)
    fake_loss = loss_fn(fake_pred, fake_labels)
    
    # Total discriminator loss
    d_loss = real_loss + fake_loss
    
    d_loss.backward()
    optimizer.step()
    
    return d_loss.item(), real_pred.mean().item(), fake_pred.mean().item()


def train_generator(batch_size, generator, discriminator, optimizer, loss_fn, latent_dim, device):
    """Train generator for one step."""
    optimizer.zero_grad()
    
    # Generate fake images
    z = torch.randn(batch_size, latent_dim).to(device)
    fake_images = generator(z)
    
    # Try to fool discriminator (want D(G(z)) = 1)
    fake_pred = discriminator(fake_images)
    real_labels = torch.ones(batch_size, 1).to(device)  # Generator wants D to predict "real"
    
    g_loss = loss_fn(fake_pred, real_labels)
    
    g_loss.backward()
    optimizer.step()
    
    return g_loss.item()


# Test the training functions
test_batch = next(iter(train_loader))[0].to(device)
test_d_loss, test_real_acc, test_fake_acc = train_discriminator(
    test_batch, generator, discriminator, optimizer_D, adversarial_loss, latent_dim, device
)
test_g_loss = train_generator(
    test_batch.size(0), generator, discriminator, optimizer_G, adversarial_loss, latent_dim, device
)

print(f"Test training step:")
print(f"  D loss: {test_d_loss:.4f}")
print(f"  G loss: {test_g_loss:.4f}")
print(f"  D(real): {test_real_acc:.4f}")
print(f"  D(fake): {test_fake_acc:.4f}")

### Understanding the Metrics:

- **D(real)**: Discriminator's output on real images (want → 1)
- **D(fake)**: Discriminator's output on fake images (want → 0)
- **D loss**: Lower is better (D is distinguishing well)
- **G loss**: Lower is better (G is fooling D)

**Healthy training:**
- D(real) ≈ 0.7-0.9
- D(fake) ≈ 0.1-0.3
- D and G losses relatively balanced

## Part 14: Training the GAN

Now let's train the DCGAN with careful monitoring of the training dynamics.

In [ ]:
# Training configuration
BATCH_SIZE = 128
NUM_EPOCHS = 50
LATENT_DIM = 100
N_CRITIC = 1  # Update discriminator every N steps
LABEL_SMOOTHING = True

# Create data loader
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=False
)

# Reinitialize models
generator = DCGANGenerator(latent_dim=LATENT_DIM).to(device)
discriminator = DCGANDiscriminator().to(device)
generator.apply(weights_init)
discriminator.apply(weights_init)

# Reinitialize optimizers
optimizer_G = optim.Adam(generator.parameters(), lr=0.0002, betas=(0.5, 0.999))
optimizer_D = optim.Adam(discriminator.parameters(), lr=0.0002, betas=(0.5, 0.999))

# Fixed noise for visualization
fixed_noise = torch.randn(64, LATENT_DIM).to(device)

print(f"GAN Training Configuration:")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Latent dim: {LATENT_DIM}")
print(f"  D update ratio: {N_CRITIC}")
print(f"  Label smoothing: {LABEL_SMOOTHING}")

In [ ]:
# Training loop
gan_history = {
    'd_loss': [],
    'g_loss': [],
    'd_real': [],
    'd_fake': []
}

print("Training GAN...\n")
start_time = time.time()

for epoch in range(NUM_EPOCHS):
    generator.train()
    discriminator.train()
    
    epoch_d_loss = 0
    epoch_g_loss = 0
    epoch_d_real = 0
    epoch_d_fake = 0
    
    for i, (real_images, _) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}", leave=False)):
        real_images = real_images.to(device)
        batch_size = real_images.size(0)
        
        # ---------------------
        # Train Discriminator
        # ---------------------
        d_loss, d_real, d_fake = train_discriminator(
            real_images, generator, discriminator, optimizer_D,
            adversarial_loss, LATENT_DIM, device, LABEL_SMOOTHING
        )
        
        # ---------------------
        # Train Generator
        # ---------------------
        if i % N_CRITIC == 0:
            g_loss = train_generator(
                batch_size, generator, discriminator, optimizer_G,
                adversarial_loss, LATENT_DIM, device
            )
        else:
            g_loss = 0
        
        # Accumulate metrics
        epoch_d_loss += d_loss
        epoch_g_loss += g_loss
        epoch_d_real += d_real
        epoch_d_fake += d_fake
    
    # Average metrics
    epoch_d_loss /= len(train_loader)
    epoch_g_loss /= len(train_loader)
    epoch_d_real /= len(train_loader)
    epoch_d_fake /= len(train_loader)
    
    # Store history
    gan_history['d_loss'].append(epoch_d_loss)
    gan_history['g_loss'].append(epoch_g_loss)
    gan_history['d_real'].append(epoch_d_real)
    gan_history['d_fake'].append(epoch_d_fake)
    
    # Print progress
    elapsed = time.time() - start_time
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} ({elapsed:.1f}s)")
    print(f"  D loss: {epoch_d_loss:.4f}, G loss: {epoch_g_loss:.4f}")
    print(f"  D(real): {epoch_d_real:.4f}, D(fake): {epoch_d_fake:.4f}")
    
    # Visualize progress
    if (epoch + 1) % 10 == 0 or epoch == 0:
        generator.eval()
        with torch.no_grad():
            fake_images = generator(fixed_noise)
        
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        
        # Generated images
        fake_grid = make_grid(fake_images * 0.5 + 0.5, nrow=8, padding=2)
        axes[0].imshow(fake_grid.permute(1, 2, 0).cpu().numpy(), cmap='gray')
        axes[0].set_title(f"Generated Images (Epoch {epoch+1})")
        axes[0].axis('off')
        
        # Training curves
        ax2 = axes[1]
        ax2.plot(gan_history['d_loss'], label='D Loss', alpha=0.7)
        ax2.plot(gan_history['g_loss'], label='G Loss', alpha=0.7)
        ax2.set_xlabel('Epoch')
        ax2.set_ylabel('Loss')
        ax2.set_title('GAN Training Progress')
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        
        # Add D(real) and D(fake) on secondary axis
        ax2_twin = ax2.twinx()
        ax2_twin.plot(gan_history['d_real'], label='D(real)', alpha=0.5, linestyle='--', color='green')
        ax2_twin.plot(gan_history['d_fake'], label='D(fake)', alpha=0.5, linestyle='--', color='red')
        ax2_twin.set_ylabel('Discriminator Output')
        ax2_twin.legend(loc='upper right')
        ax2_twin.set_ylim([0, 1])
        
        plt.tight_layout()
        plt.show()

total_time = time.time() - start_time
print(f"\nTraining complete in {total_time:.1f}s ({total_time/60:.1f} minutes)")

### Training Observations:

**Signs of healthy GAN training:**
1. **D(real) ≈ 0.7-0.9**: D correctly identifies real images
2. **D(fake) ≈ 0.1-0.3**: D correctly identifies fake images
3. **Both losses decrease**: Both networks are learning
4. **Generated quality improves**: Visual inspection shows progress

**Warning signs:**
- **D(fake) → 0 too quickly**: D is too strong, G can't learn
- **D(real) → 0.5, D(fake) → 0.5**: D collapsed, not learning
- **Mode collapse**: All generated images look similar
- **Training oscillation**: Metrics bounce wildly

## Part 15: Comparing VAE vs GAN

Let's generate samples from both models and compare the results.

In [ ]:
# Generate samples from both models
vae.eval()
generator.eval()

with torch.no_grad():
    # VAE samples
    vae_samples = vae.generate(64)
    
    # GAN samples
    z = torch.randn(64, LATENT_DIM).to(device)
    gan_samples = generator(z)

# Display side by side
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# VAE samples
vae_grid = make_grid(vae_samples * 0.5 + 0.5, nrow=8, padding=2)
axes[0].imshow(vae_grid.permute(1, 2, 0).cpu().numpy(), cmap='gray')
axes[0].set_title("VAE Generated Samples", fontsize=14, fontweight='bold')
axes[0].axis('off')

# GAN samples
gan_grid = make_grid(gan_samples * 0.5 + 0.5, nrow=8, padding=2)
axes[1].imshow(gan_grid.permute(1, 2, 0).cpu().numpy(), cmap='gray')
axes[1].set_title("GAN Generated Samples", fontsize=14, fontweight='bold')
axes[1].axis('off')

plt.tight_layout()
plt.show()

### VAE vs GAN Comparison

| Aspect | VAE | GAN |
|--------|-----|-----|
| **Image Quality** | Blurry, averaged | Sharp, realistic |
| **Training Stability** | Stable, consistent | Difficult, requires tuning |
| **Mode Coverage** | Covers all modes | May suffer mode collapse |
| **Latent Space** | Structured, interpretable | Less structured |
| **Evaluation** | ELBO (principled metric) | Difficult to evaluate |
| **Speed** | Fast convergence | Slower, may need many epochs |
| **Overfitting** | Less prone | More prone (D can memorize) |
| **Applications** | Compression, anomaly detection | High-quality generation |

### When to Use Each:

**Use VAE when:**
- Need stable training
- Want interpretable latent space
- Need likelihood estimates
- Limited computational resources

**Use GAN when:**
- Need highest quality images
- Can afford extensive hyperparameter tuning
- Have sufficient training data
- Visual quality is most important

## Part 16: Conditional Generation

Both VAE and GAN can be extended to **conditional generation**: controlling what class of digit to generate.

### Idea:
- Provide class label $y$ as additional input
- Model learns $P(X|Y)$ instead of $P(X)$
- At generation time, specify desired class

For simplicity, we'll demonstrate a basic conditional GAN (cGAN) approach.

In [ ]:
class ConditionalGenerator(nn.Module):
    """Conditional GAN Generator: (z, label) -> image"""
    
    def __init__(self, latent_dim=100, num_classes=10, embed_dim=10):
        super().__init__()
        self.latent_dim = latent_dim
        self.num_classes = num_classes
        
        # Embedding for labels
        self.label_embedding = nn.Embedding(num_classes, embed_dim)
        
        # Combine latent vector and label embedding
        input_dim = latent_dim + embed_dim
        
        # Same architecture as DCGAN but with concatenated input
        self.fc = nn.Linear(input_dim, 256 * 7 * 7)
        
        self.deconv_layers = nn.Sequential(
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(True),
            nn.Conv2d(64, 1, kernel_size=3, stride=1, padding=1),
            nn.Tanh()
        )
    
    def forward(self, z, labels):
        # Embed labels
        label_embed = self.label_embedding(labels)
        
        # Concatenate latent vector and label embedding
        z_combined = torch.cat([z, label_embed], dim=1)
        
        # Generate image
        h = self.fc(z_combined)
        h = h.view(h.size(0), 256, 7, 7)
        img = self.deconv_layers(h)
        
        return img


# Create conditional generator
cond_gen = ConditionalGenerator(latent_dim=100, num_classes=10).to(device)
cond_gen.apply(weights_init)

print("Conditional generator created")
print(f"Parameters: {sum(p.numel() for p in cond_gen.parameters()):,}")

### Generating Specific Digits

Even without training the conditional model (which would take significant time), we can demonstrate the concept:

**With a trained conditional GAN:**
1. Sample noise $z \sim \mathcal{N}(0, I)$
2. Choose desired digit class $y$ (e.g., $y = 3$)
3. Generate: $x = G(z, y)$
4. Result: Image of digit "3"

This allows **controlled generation** - very useful for practical applications!

In [ ]:
# Demonstrate conditional generation interface (even though model is untrained)
cond_gen.eval()

with torch.no_grad():
    # Generate 10 samples for each digit class (0-9)
    samples_per_class = 10
    all_samples = []
    
    for digit in range(10):
        # Sample noise
        z = torch.randn(samples_per_class, 100).to(device)
        
        # Create labels (all same digit)
        labels = torch.full((samples_per_class,), digit, dtype=torch.long).to(device)
        
        # Generate
        samples = cond_gen(z, labels)
        all_samples.append(samples)
    
    all_samples = torch.cat(all_samples, dim=0)

# Note: these won't look good since model is untrained!
# Just demonstrating the interface
print("Generated conditional samples (untrained model - will be noise)")
print("In practice, after training, each row would show samples for digits 0-9")

grid = make_grid(all_samples[:50] * 0.5 + 0.5, nrow=10, padding=2)
plt.figure(figsize=(14, 8))
plt.imshow(grid.permute(1, 2, 0).cpu().numpy(), cmap='gray')
plt.title("Conditional Generation Demo (Untrained - Shows Concept Only)")
plt.axis('off')
plt.show()

## Part 17: Latent Space Manipulation (GAN)

Let's explore the GAN's latent space with interpolation and vector arithmetic.

In [ ]:
def interpolate_gan(generator, z1, z2, num_steps=10):
    """Interpolate between two latent vectors."""
    generator.eval()
    
    with torch.no_grad():
        # Create interpolation steps
        alphas = torch.linspace(0, 1, num_steps).to(device)
        interpolations = []
        
        for alpha in alphas:
            # Linear interpolation
            z_interp = (1 - alpha) * z1 + alpha * z2
            
            # Generate
            img = generator(z_interp)
            interpolations.append(img)
        
        return torch.cat(interpolations, dim=0)


# Sample two random latent vectors
z1 = torch.randn(1, LATENT_DIM).to(device)
z2 = torch.randn(1, LATENT_DIM).to(device)

# Interpolate
interpolated = interpolate_gan(generator, z1, z2, num_steps=12)

show_images(interpolated, "GAN Latent Space Interpolation", nrow=12, figsize=(15, 3))

In [ ]:
# Multiple interpolations
fig, axes = plt.subplots(4, 1, figsize=(15, 12))

for i in range(4):
    z1 = torch.randn(1, LATENT_DIM).to(device)
    z2 = torch.randn(1, LATENT_DIM).to(device)
    
    interpolated = interpolate_gan(generator, z1, z2, num_steps=12)
    grid = make_grid(interpolated * 0.5 + 0.5, nrow=12, padding=2)
    
    axes[i].imshow(grid.permute(1, 2, 0).cpu().numpy(), cmap='gray')
    axes[i].set_title(f"Interpolation {i+1}")
    axes[i].axis('off')

plt.suptitle("GAN Latent Space Interpolations", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Latent Vector Arithmetic

GANs can learn semantically meaningful directions in latent space.

**Famous example (from faces):**
- Vector("King") - Vector("Man") + Vector("Woman") ≈ Vector("Queen")

For MNIST, we might find directions like:
- Rotation angle
- Stroke thickness  
- Slant/italic

In [ ]:
# Explore random directions in latent space
generator.eval()

with torch.no_grad():
    # Start from a random point
    z_base = torch.randn(1, LATENT_DIM).to(device)
    
    # Random direction
    direction = torch.randn(1, LATENT_DIM).to(device)
    direction = direction / direction.norm()  # Normalize
    
    # Walk along direction
    steps = torch.linspace(-4, 4, 12).to(device)
    walk = []
    
    for step in steps:
        z = z_base + step * direction
        img = generator(z)
        walk.append(img)
    
    walk_images = torch.cat(walk, dim=0)

show_images(walk_images, "Random Walk in GAN Latent Space", nrow=12, figsize=(15, 3))

## Part 18: Common Training Challenges

Let's discuss and demonstrate common issues in generative modeling.

### Challenge 1: Mode Collapse (GAN)

**Problem**: Generator produces limited variety (e.g., only digit "1")

**Causes:**
- G finds an "easy" sample that fools D
- G exploits weaknesses in D instead of learning full distribution

**Solutions:**
1. **Minibatch discrimination**: Let D compare samples within batch
2. **Feature matching**: Match statistics of real/fake in intermediate layers
3. **Unrolled GANs**: Update G considering future D updates
4. **Training tricks**: Label smoothing, update ratio adjustments
5. **Architecture changes**: Use WGAN or other variants

**Detection**: Check if generated samples lack diversity

### Challenge 2: Posterior Collapse (VAE)

**Problem**: KL divergence → 0, model ignores latent code

**Causes:**
- Decoder too powerful (can reconstruct without using z)
- KL term dominates loss too early

**Solutions:**
1. **KL annealing**: Gradually increase KL weight from 0 to 1
2. **Free bits**: Allow some minimum KL per dimension
3. **Weaker decoder**: Limit decoder capacity
4. **Beta-VAE**: Tune β parameter

**Detection**: Monitor KL divergence, check if latent space affects outputs

### Challenge 3: Training Instability (GAN)

**Problem**: Training diverges, loss oscillates wildly

**Causes:**
- Imbalanced G/D strength
- Poor initialization
- Learning rate too high

**Solutions:**
1. **Learning rate**: Use lower LR (e.g., 0.0002)
2. **Update ratio**: Update D more frequently
3. **Gradient penalties**: Add regularization (WGAN-GP)
4. **Batch normalization**: Stabilizes training
5. **Spectral normalization**: Constrains discriminator

**Detection**: Monitor D(real), D(fake), check if one stays near 0 or 1

### Challenge 4: Blurry Images (VAE)

**Problem**: VAE produces blurry reconstructions

**Causes:**
- MSE loss averages over possible outputs
- Probabilistic nature leads to uncertainty

**Solutions:**
1. **Perceptual loss**: Use feature-based loss instead of MSE
2. **Adversarial loss**: Add discriminator (VAE-GAN hybrid)
3. **Discrete latents**: Use VQ-VAE
4. **Higher capacity**: Larger model, more latent dims

**Trade-off**: Sharpness vs training stability

## Part 19: Evaluation Metrics

How do we measure generation quality objectively?

### VAE Metrics:

1. **ELBO (Evidence Lower Bound)**
   - Principled probabilistic metric
   - Lower is better (negative log-likelihood)
   - Decomposed: Reconstruction + KL

2. **Reconstruction Error**
   - MSE or binary cross-entropy
   - Measures fidelity to training data

3. **Latent Space Organization**
   - Visualize latent codes (t-SNE, PCA)
   - Check if classes cluster
   - Measure interpolation smoothness

### GAN Metrics:

1. **Inception Score (IS)**
   - Uses pre-trained classifier
   - Measures quality and diversity
   - Higher is better
   - Formula: $\exp(\mathbb{E}_x D_{KL}(p(y|x) \| p(y)))$

2. **Frechet Inception Distance (FID)**
   - Compares feature distributions
   - Lower is better
   - More robust than IS
   - Currently most popular metric

3. **Mode Score**
   - Measures mode coverage
   - Detects mode collapse

4. **Visual Inspection**
   - Still important!
   - Check for artifacts, diversity, realism

### Computing FID (Simplified):

```python
# 1. Extract features from real and fake images using Inception network
real_features = inception_model(real_images)
fake_features = inception_model(fake_images)

# 2. Compute mean and covariance
mu_real, sigma_real = real_features.mean(0), cov(real_features)
mu_fake, sigma_fake = fake_features.mean(0), cov(fake_features)

# 3. Compute Frechet distance
fid = ||mu_real - mu_fake||^2 + Tr(sigma_real + sigma_fake - 2*sqrt(sigma_real @ sigma_fake))
```

**Note**: For MNIST, these metrics are less meaningful (designed for natural images). Visual inspection is most important.

## Part 20: Experiments and Extensions

Here are ideas for further exploration:

### Experiment 1: Beta-VAE

**Task**: Train VAE with different β values (0.5, 1.0, 2.0, 5.0)

**Questions:**
- How does reconstruction quality change?
- How does latent space structure change?
- Is there a sweet spot for β?

**Expected results:**
- Lower β: Sharper images, less structured latent
- Higher β: Blurrier images, more disentangled latent

### Experiment 2: Latent Dimension

**Task**: Compare VAE/GAN with different latent dimensions (5, 10, 20, 50, 100)

**Questions:**
- What's the minimum dimension for good quality?
- Does higher dimension always improve quality?
- How does training time scale?

**Expected results:**
- Too low: Can't capture all variation
- Sweet spot: ~20-50 for MNIST
- Too high: Overfitting, slower training

### Experiment 3: Architecture Variants

**Task**: Try different architectures
- Fully connected (MLP) instead of convolutional
- Deeper networks (more layers)
- ResNet-style skip connections

**Questions:**
- How important are convolutions for images?
- Do deeper networks always help?
- Can we make training more stable?

**Expected results:**
- Convolutions crucial for image structure
- Depth helps but diminishing returns
- Skip connections can stabilize

### Experiment 4: Loss Functions

**Task**: Try alternative loss functions
- VAE: Binary cross-entropy instead of MSE
- GAN: Wasserstein loss (WGAN)
- GAN: Least squares loss (LSGAN)

**Questions:**
- How does loss function affect image quality?
- Which is most stable to train?
- Are there convergence differences?

**Expected results:**
- BCE can work better for binary images
- Wasserstein more stable but complex
- LS-GAN can be easier to train

### Experiment 5: Other Datasets

**Task**: Apply models to other datasets
- Fashion-MNIST (clothing items)
- CIFAR-10 (color images)
- CelebA (faces)

**Questions:**
- Do techniques generalize?
- What needs to change for color images?
- How does complexity affect results?

**Expected results:**
- Fashion-MNIST similar to MNIST
- CIFAR-10 needs bigger models
- CelebA very challenging, needs advanced techniques

## Part 21: Summary and Key Takeaways

### What We Learned:

#### 1. Generative Modeling Fundamentals
- **Goal**: Learn $P(X)$ to generate new samples
- **Two approaches**: Explicit (VAE) vs Implicit (GAN)
- **Key concept**: Latent variable models

#### 2. Variational Autoencoders (VAE)
- **Architecture**: Encoder (x → z) + Decoder (z → x)
- **Training**: Maximize ELBO = Reconstruction - KL
- **Innovation**: Reparameterization trick enables backprop
- **Strengths**: Stable training, structured latent space
- **Weaknesses**: Blurry images, averaged outputs

#### 3. Generative Adversarial Networks (GAN)
- **Architecture**: Generator (z → x) + Discriminator (x → real/fake)
- **Training**: Minimax game, alternating updates
- **Innovation**: Adversarial training signal
- **Strengths**: Sharp images, high quality
- **Weaknesses**: Training instability, mode collapse

#### 4. DCGAN Best Practices
- Strided convolutions (no pooling)
- Batch normalization (except input/output)
- LeakyReLU (discriminator), ReLU + Tanh (generator)
- Adam optimizer (lr=0.0002, β₁=0.5)
- Label smoothing and careful tuning

#### 5. Latent Space Manipulation
- **Interpolation**: Smooth transitions between samples
- **Arithmetic**: Semantic operations in latent space
- **Conditional generation**: Control output attributes

#### 6. Training Challenges
- **Mode collapse**: Generator produces limited variety
- **Posterior collapse**: VAE ignores latent code
- **Instability**: Diverging losses, oscillations
- **Blurriness**: VAE averaging effect

### Practical Guidance:

**Starting a new project?**
1. Start with VAE for stability and fast iteration
2. If quality insufficient, try GAN with DCGAN architecture
3. Use conditional variants if need control
4. Monitor training carefully, be patient
5. Visual inspection is crucial!

**Debugging tips:**
- VAE: Check KL doesn't collapse to 0
- GAN: Monitor D(real) and D(fake), should stay balanced
- Both: Visualize samples frequently during training
- Save checkpoints regularly

### Further Reading:

**Papers:**
- VAE: "Auto-Encoding Variational Bayes" (Kingma & Welling, 2014)
- GAN: "Generative Adversarial Networks" (Goodfellow et al., 2014)
- DCGAN: "Unsupervised Representation Learning with Deep Convolutional GANs" (Radford et al., 2015)
- Beta-VAE: "beta-VAE: Learning Basic Visual Concepts with a Constrained Variational Framework" (Higgins et al., 2017)

**Advanced Topics:**
- Normalizing Flows (invertible transformations)
- Diffusion Models (iterative denoising)
- VQ-VAE (discrete latents)
- StyleGAN (style-based generation)
- Conditional GANs (pix2pix, CycleGAN)
- Stable Diffusion (text-to-image)

### Next Steps:

1. **Experiment**: Try the suggested experiments above
2. **Extend**: Implement conditional VAE fully
3. **Scale up**: Try on CIFAR-10 or Fashion-MNIST
4. **Advanced models**: Implement WGAN, Progressive GAN
5. **Applications**: Use for data augmentation, anomaly detection

### Final Thoughts:

Generative modeling is a rapidly evolving field. While VAEs and GANs are foundational, newer approaches like diffusion models are pushing state-of-the-art. However, understanding VAE/GAN principles provides crucial intuition for all generative modeling.

**Remember**: 
- No single approach is "best" - depends on your application
- Training generative models requires patience and iteration
- Visual quality matters, but so does diversity and coverage
- The field is moving fast - stay curious and keep learning!

## Reflection Questions:

1. **Understanding**: Can you explain VAE and GAN training in your own words?
2. **Trade-offs**: When would you choose VAE over GAN, or vice versa?
3. **Debugging**: If your GAN training diverged, what would you try first?
4. **Applications**: What real-world problem could you solve with generative models?
5. **Extensions**: How would you modify these models for color images?
6. **Ethics**: What are potential misuses of generative models? How can we address them?

Take time to think through these questions - understanding the "why" is more important than memorizing the "what"!